# F1 Score - Setup 

In [15]:
# Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [16]:
# Load Data
df = pd.read_csv('../data/raw/flood_data.csv')

# Display first few rows
print(df.head())
print(f"\nDataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")

   rainfall  temperature  humidity  river_level  soil_moisture  flood_risk
0      23.5         18.2      45.0          1.2           0.35           0
1      34.2         22.1      52.0          1.8           0.42           0
2      45.6         25.3      65.0          2.5           0.48           1
3      28.3         20.5      48.0          1.5           0.38           0
4      52.1         28.1      72.0          3.2           0.55           1

Dataset shape: (51, 6)

Column names: ['rainfall', 'temperature', 'humidity', 'river_level', 'soil_moisture', 'flood_risk']


In [17]:
# Prepare Data - Assume last column is target
# Adjust column names based on your actual data
X = df.iloc[:, :-1]  # All columns except last
y = df.iloc[:, -1]   # Last column is target

# Handle missing values
X = X.fillna(X.mean())

# Split into Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")
print(f"y_test distribution:\n{y_test.value_counts()}")

X_train shape: (40, 5)
X_test shape: (11, 5)
y_train distribution:
flood_risk
0    21
1    19
Name: count, dtype: int64
y_test distribution:
flood_risk
1    6
0    5
Name: count, dtype: int64


In [18]:
# Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [19]:
# Make Predictions
y_pred = model.predict(X_test_scaled)

print(f"Predictions shape: {y_pred.shape}")
print(f"Unique predictions: {np.unique(y_pred)}")
print("Ready for F1 Score evaluation!")

Predictions shape: (11,)
Unique predictions: [0 1]
Ready for F1 Score evaluation!


## F1 Score Evaluation

Now we evaluate the model's performance using various F1 score metrics.

In [20]:
#Import Libraries
from sklearn.metrics import f1_score, classification_report

In [21]:
#Compute F1 Score
f1 = f1_score(y_test, y_pred)

print(f"F1-Score: {f1:.3f}")

F1-Score: 1.000


In [22]:
#Full Classification Report
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      1.000     1.000     1.000         5
           1      1.000     1.000     1.000         6

    accuracy                          1.000        11
   macro avg      1.000     1.000     1.000        11
weighted avg      1.000     1.000     1.000        11



In [23]:
#Comparing Against Baseline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("=== Baseline ===")
print(f"F1: {f1_score(y_test, baseline_pred, zero_division=0):.3f}")

print("=== Model ===")
print(f"F1: {f1_score(y_test, y_pred):.3f}")

=== Baseline ===
F1: 0.000
=== Model ===
F1: 1.000


In [24]:
#Multi Class F1 Variants

#Macro F1
f1_macro = f1_score(y_test, y_pred, average="macro")

#Micro F1
f1_micro = f1_score(y_test, y_pred, average="micro")

#Weighted F1
f1_weighted = f1_score(y_test, y_pred, average="weighted")

In [25]:
#Threshold Optimization for F1

#Step 1 — Get Prediction Probabilities
y_prob = model.predict_proba(X_test)[:, 1]

#Step 2 — Search Best Threshold
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)

f1_scores = [
    f1_score(y_test, (y_prob >= t).astype(int))
    for t in thresholds
]

best_threshold = thresholds[np.argmax(f1_scores)]

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best F1: {max(f1_scores):.3f}")

Best Threshold: 0.10
Best F1: 0.706


c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [26]:
#Cross validation with F1
from sklearn.model_selection import cross_val_score
import numpy as np

cv_f1 = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)

print(f"CV F1 Scores: {cv_f1.round(3)}")
print(f"Mean CV F1: {cv_f1.mean():.3f} ± {cv_f1.std():.3f}")

CV F1 Scores: [1. 1. 1. 1. 1.]
Mean CV F1: 1.000 ± 0.000
